## Environment Execution Runtimes (Local vs. Subprocess vs. Containerized / Docker)

## 1. The Three Execution Runtime Models

OpenEnv is designed to support three distinct execution backend architectures depending on development speed, isolation guarantees, and training scale:

### Runtime options

1. **Local In-Process Runtime**
2. **Subprocess Runtime**
3. **Containerized / Client-Server Runtime (OpenEnv Standard)**

### Architecture diagram

**Option 1: Local In-Process**
```
Python Training Loop (Same PID / Memory)
  └── In-Memory REPL
```

**Option 2: Subprocess**
```
Python Training Loop (Master PID)
  └── Python Subprocess (Isolated PID)
```

**Option 3: Containerized / Client-Server**
```
Python Training Process (TRL / Client)
  └── Sync / Async EnvClient via WebSocket/HTTP
        │
        └── Container Sandbox (Docker / FastAPI Server)
              • Fresh filesystem, dedicated dependencies
              • Real shell, browser, or tool runner
```

## 2. Runtime Deep-Dive and Tradeoffs

### Comparison table

| Runtime Architecture | How it Executes | Isolation Level | Latency per Step | Failure Modes & Production Risks |
|---|---|---|---|---|
| Local In-Process | Direct execution in Python memory via `exec()` or simple state machines (for example, TextArena games) | None. Runs in the exact same process as PyTorch | Microseconds ($\mu\text{s}$). Fastest possible throughput | State pollution: global variables leak across episodes; process crash: a memory leak or segfault crashes the entire training job |
| Local Subprocess | Parent training process spawns a worker process via `subprocess.Popen` or `multiprocessing` | Process-level. Memory is isolated; filesystem and OS are shared | Milliseconds ($5\text{–}50\text{ ms}$) | Host damage: the agent can write to host files or kill host processes; orphaned processes: zombie processes accumulate if unhandled |
| Containerized / Docker (OpenEnv Native) | The environment runs as a standalone server (FastAPI/WebSocket) inside an isolated Docker container. The RL loop connects via an `EnvClient` | Full OS / filesystem isolation. The container can be wiped or reset cleanly | Higher ($10\text{–}100\text{ ms}$ local, depending on network / I/O) | Docker daemon overhead: managing hundreds of active containers in parallel rollouts requires container orchestration such as Docker Swarm or Kubernetes |

## 3. The OpenEnv Client-Server Architecture

In production agent RL (such as SWE-bench code repair, terminal automation, or multi-tool workflows), OpenEnv uses a client-server decoupled design:

```text
┌────────────────────────────────────────────────────────────────────────┐
│ TRAINING WORKER (GPU)                                                │
│ TRL / GRPO Trainer                                                   │
│                                                                          │
│ Generates Text Action                                                 │
│ SyncEnvClient / EnvClient                                             │
│                                                                          │
│ HTTP/WebSocket: POST /step {"command": "pytest"}                      │
│ Response: {"obs": "...", "reward": 0.0, "done": false}             │
└───────────────────────┬────────────────────────────────────────────────┘
                        │
                        ▼
┌────────────────────────────────────────────────────────────────────────┐
│ DOCKER CONTAINER (SANDBOX RUNTIME)                                     │
│ FastAPI / Uvicorn Server                                              │
│ Executes command in isolated shell                                    │
│ Captures stdout/stderr & exit code                                    │
│ Returns Observation & StepResult                                      │
└────────────────────────────────────────────────────────────────────────┘
```

## Why Decouple the Environment from the GPU Training Process?

- **Host safety**: untrusted agent-generated code runs strictly within a locked container sandbox with restricted root privileges and network policies.
- **Determinism on reset()**: when `env.reset()` is invoked, the container can instantly revert the workspace (for example, via `git reset --hard` or container overlay filesystem resets) to ensure pristine reproducibility for the next episode.
- **Disaggregated scaling**: your GPU machines focus entirely on LLM inference and backpropagation, while environment rollouts can be distributed across a pool of CPU container workers.